## Example 1: Markdown of a package's public symbols

In [ ]:
import sys
import tempfile
from pathlib import Path

d = Path(tempfile.mkdtemp())
(d / "toypkg").mkdir()
(d / "toypkg" / "__init__.py").write_text('"""A toy package."""\n')
(d / "toypkg" / "shapes.py").write_text(
    'class Circle:\n    """A circle."""\n    def area(self, scale: float = 1.0) -> float:\n        """Area of the circle."""\n        return 3.14 * scale\n'
    'def make(n: int) -> list:\n    """Make n circles."""\n    return [Circle() for _ in range(n)]\n'
    'def _hidden():\n    """Private."""\n')
sys.path.insert(0, str(d))
for _m in [m for m in sys.modules if m.split('.')[0] == 'toypkg']:
    del sys.modules[_m]  # a fresh toypkg per example

import importlib.metadata

from pysym2md.core import generate_markdown

# Example 1: classes, methods and functions with their signatures and docstrings; private names are left out
assert importlib.metadata.version("pysym2md") == "0.1.2"
md = generate_markdown("toypkg", include_no_docstring=False)
assert "toypkg.shapes" in md and "Circle" in md and "def area" in md and "Area of the circle." in md, md
assert "Make n circles." in md and "_hidden" not in md, md
print("Example 1 OK:\n" + md[:600])

## Example 2: Symbols without docstrings are opt-in

In [ ]:
import sys
import tempfile
from pathlib import Path

d = Path(tempfile.mkdtemp())
(d / "toypkg").mkdir()
(d / "toypkg" / "__init__.py").write_text('"""A toy package."""\n')
(d / "toypkg" / "shapes.py").write_text(
    'class Circle:\n    """A circle."""\n    def area(self, scale: float = 1.0) -> float:\n        """Area of the circle."""\n        return 3.14 * scale\n'
    'def make(n: int) -> list:\n    """Make n circles."""\n    return [Circle() for _ in range(n)]\n'
    'def _hidden():\n    """Private."""\n')
sys.path.insert(0, str(d))
for _m in [m for m in sys.modules if m.split('.')[0] == 'toypkg']:
    del sys.modules[_m]  # a fresh toypkg per example

from pysym2md.core import generate_markdown

# Example 2: include_no_docstring=True also lists undocumented public symbols
(d / "toypkg" / "bare.py").write_text("def nodoc(x):\n    return x\n")
assert "nodoc" not in generate_markdown("toypkg", include_no_docstring=False)
assert "nodoc" in generate_markdown("toypkg", include_no_docstring=True)
print("Example 2 OK")

## Example 3: The pysym2md command writes a file

In [ ]:
import sys
import tempfile
from pathlib import Path

d = Path(tempfile.mkdtemp())
(d / "toypkg").mkdir()
(d / "toypkg" / "__init__.py").write_text('"""A toy package."""\n')
(d / "toypkg" / "shapes.py").write_text(
    'class Circle:\n    """A circle."""\n    def area(self, scale: float = 1.0) -> float:\n        """Area of the circle."""\n        return 3.14 * scale\n'
    'def make(n: int) -> list:\n    """Make n circles."""\n    return [Circle() for _ in range(n)]\n'
    'def _hidden():\n    """Private."""\n')
sys.path.insert(0, str(d))
for _m in [m for m in sys.modules if m.split('.')[0] == 'toypkg']:
    del sys.modules[_m]  # a fresh toypkg per example

import os, shutil, subprocess

# Example 3: pysym2md <package> --output-file F (fastcore 2.x call_parse spells the options with hyphens)
exe = shutil.which("pysym2md")
assert exe, "pysym2md is not on PATH"
out = d / "symbols.md"
env = {**os.environ, "PYTHONPATH": str(d)}
r = subprocess.run([exe, "toypkg", "--output-file", str(out)], capture_output=True, text=True, env=env)
assert r.returncode == 0 and "Circle" in out.read_text(), (r.stdout, r.stderr)
print("Example 3 OK:", out.stat().st_size, "bytes")

## Example 4: A manifest of a real installed package

In [ ]:
from pysym2md.core import generate_markdown

md = generate_markdown("check_wheel_contents", include_no_docstring=False)
assert "WheelChecker" in md and "check_wheel_contents.checker" in md, md[:300]
print("Example 4 OK:", len(md.splitlines()), "lines")

## Example 5: The building blocks

In [ ]:
from pysym2md.core import format_symbol, is_public_symbol

# Example 5: which names count as public, and how one symbol is formatted
assert is_public_symbol("make") and is_public_symbol("__init__") and not is_public_symbol("_hidden")
text = format_symbol("make", "(n: int) -> list", "Make n circles.")
assert "make" in text and "Make n circles." in text, text
print("Example 5 OK:\n" + text)